In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import scipy
import statsmodels.api as sm

In [2]:
from scipy.stats import mannwhitneyu
import scipy.stats as stats


In [3]:
# Загрузим данные и оставим только пользователей, у кого ОС Андроид.
taxi = pd.read_csv('/content/new_dataframe.csv')
taxi.head()

,Unnamed: 0,user_id,hour,os,order_class,surge,app_opened,price_seen,order_made,ride_completed,user_cancelled,city_center_order,distance,age,rfm
0,0,867689,12,iOS,business,no surge,1,1,1,1,0,0,7.982135,20,low
1,1,752172,5,Android,economy,no surge,1,1,1,1,0,1,2.908468,27,high
2,2,486559,15,Android,comfort,no surge,1,1,1,1,0,0,7.224614,21,high
3,3,304024,0,Android,economy,no surge,1,1,1,1,0,1,1.874349,52,low
4,4,139420,0,Android,business,no surge,1,1,1,1,0,0,10.704778,19,low


In [4]:
taxi_1 = taxi[taxi['os'] == 'Android']
pd.DataFrame(taxi_1)
taxi_1.head()

,Unnamed: 0,user_id,hour,os,order_class,surge,app_opened,price_seen,order_made,ride_completed,user_cancelled,city_center_order,distance,age,rfm
1,1,752172,5,Android,economy,no surge,1,1,1,1,0,1,2.908468,27,high
2,2,486559,15,Android,comfort,no surge,1,1,1,1,0,0,7.224614,21,high
3,3,304024,0,Android,economy,no surge,1,1,1,1,0,1,1.874349,52,low
4,4,139420,0,Android,business,no surge,1,1,1,1,0,0,10.704778,19,low
5,5,139455,5,Android,comfort,NaN,1,0,0,0,0,1,NaN,24,high


In [5]:
# Оставим только тех пользователей, что завершили поездку удачно.
taxi_1['ride_completed'].replace('1', '1.0', inplace=True)
taxi_1['ride_completed'] = taxi_1['ride_completed'].astype(float)

taxi_2 = taxi_1[taxi_1['ride_completed'] == 1.0]
pd.DataFrame(taxi_2)
taxi_2.head()

/tmp/ipython-input-3737891059.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  taxi_1['ride_completed'].replace('1', '1.0', inplace=True)
/tmp/ipython-input-3737891059.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  taxi_1['ride_completed'].replace('1', '1.0', inplace=True)
/tmp/ipython-input-3737891059.py:3: SettingWithCopyWarning: 
A 

,Unnamed: 0,user_id,hour,os,order_class,surge,app_opened,price_seen,order_made,ride_completed,user_cancelled,city_center_order,distance,age,rfm
1,1,752172,5,Android,economy,no surge,1,1,1,1.0,0,1,2.908468,27,high
2,2,486559,15,Android,comfort,no surge,1,1,1,1.0,0,0,7.224614,21,high
3,3,304024,0,Android,economy,no surge,1,1,1,1.0,0,1,1.874349,52,low
4,4,139420,0,Android,business,no surge,1,1,1,1.0,0,0,10.704778,19,low
7,7,682337,2,Android,comfort,no surge,1,1,1,1.0,0,1,9.055344,21,low


In [6]:
# Создадим колонку с разбивкой на группу №1 и группу №0 и добавим эту колонку к датафрейму.
n = 101499
binary = pd.DataFrame()
binary["Binary"] = np.random.choice([0, 1], size=n)
binary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101499 entries, 0 to 101498
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   Binary  101499 non-null  int64
dtypes: int64(1)
memory usage: 793.1 KB


In [7]:
# Подкорректируем для дальнейшего анализа.
taxi_2['binary'] = binary
taxi_2.info()

taxi_2['binary'] = taxi_2['binary'].fillna(1.0)
taxi_2['binary'] = taxi_2['binary'].astype(float)
taxi_2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31448 entries, 1 to 101499
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         31448 non-null  int64  
 1   user_id            31448 non-null  int64  
 2   hour               31448 non-null  int64  
 3   os                 31448 non-null  object 
 4   order_class        31448 non-null  object 
 5   surge              31448 non-null  object 
 6   app_opened         31448 non-null  int64  
 7   price_seen         31448 non-null  int64  
 8   order_made         31448 non-null  int64  
 9   ride_completed     31448 non-null  float64
 10  user_cancelled     31448 non-null  int64  
 11  city_center_order  31448 non-null  int64  
 12  distance           31448 non-null  float64
 13  age                31448 non-null  int64  
 14  rfm                31448 non-null  object 
 15  binary             31447 non-null  float64
dtypes: float64(3), int64(9), o

/tmp/ipython-input-4013196365.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  taxi_2['binary'] = binary
/tmp/ipython-input-4013196365.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  taxi_2['binary'] = taxi_2['binary'].fillna(1.0)
/tmp/ipython-input-4013196365.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_g

In [8]:
# Оставим нужные колонки, разобъем на две отдельные выборки.
taxi_3 = taxi_2[['app_opened', 'binary']]
pd.DataFrame(taxi_3)

group_1 = taxi_3.loc[taxi_2['binary'] == 1.0]
pd.DataFrame(group_1)

group_11 = group_1[['app_opened']]
group_2 = taxi_3.loc[taxi_2['binary'] == 0.0]
pd.DataFrame(group_2)
group_22 = group_2[['app_opened']]
group_22.head()

,app_opened
3,1
4,1
16,1
28,1
32,1


In [9]:
# Проведем т-тест для двух выборок.
stat, p = mannwhitneyu(group_11, group_22, use_continuity =True)

print('Uкр=%.3f, p-value=%.3f' % (stat, p))
alpha = 0.05

if stat >= alpha:
     print("Гипотезу 𝐻1 (альтернативную) принимаем >> Различия являются статистически достоверными."),
else:
     print("Принимаем 𝐻0 гипотезу >> Различия не являются статистически достоверными и носят случайный характер.")

Uкр=123619423.500, p-value=1.000
Гипотезу 𝐻1 (альтернативную) принимаем >> Различия являются статистически достоверными.


/tmp/ipython-input-404549467.py:4: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print('Uкр=%.3f, p-value=%.3f' % (stat, p))
